# Report explainability figures

Generates explainability figures from CSV outputs produced by the explainability step.

Outputs:
- Figure 8: permutation importance
- Figure 9: local SHAP contribution waterfall

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pipeline.py").exists() and (candidate / "models").exists():
            return candidate
    raise FileNotFoundError("Could not find project root.")


PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / "csv_analysis" / "explainability_figures"
PROJECT_ROOT, OUTPUT_DIR

## Figure 8

Global permutation importance measured as macro F1 decrease after shuffling each feature.

In [ ]:
importance = pd.read_csv(OUTPUT_DIR / "figure_8_permutation_importance.csv")
plot_df = importance.head(20).sort_values("importance_mean", ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=plot_df, x="importance_mean", y="feature", color="#3b82f6", ax=ax)
ax.set_title("Global Feature Importance - Permutation Importance")
ax.set_xlabel("Macro F1 decrease after permutation")
ax.set_ylabel("Feature")
sns.despine(left=True)
plt.tight_layout()
output_path = OUTPUT_DIR / "figure_8_permutation_importance.png"
plt.savefig(output_path, dpi=300)
plt.show()
plt.close()
print(f"Saved to: {output_path}")

## Figure 9

Local SHAP contribution plot for the selected high-confidence misclassification. The CSV contains feature-level SHAP values; this notebook recreates a compact waterfall-style view from those values.

In [ ]:
shap_values = pd.read_csv(OUTPUT_DIR / "figure_9_local_shap_values.csv")
metadata = pd.read_csv(OUTPUT_DIR / "figure_9_local_shap_metadata.csv").iloc[0]

import shap

single = shap.Explanation(
    values=shap_values["shap_value"].to_numpy(),
    base_values=float(metadata["base_value"]),
    data=shap_values["feature_value"].to_numpy(),
    feature_names=shap_values["feature"].tolist(),
)

shap.plots.waterfall(single, max_display=15, show=False)
plt.title(
    f"Local SHAP Explanation - predicted {metadata['predicted_label']}, "
    f"true {metadata['true_label']}"
)
plt.tight_layout()
output_path = OUTPUT_DIR / "figure_9_local_shap_waterfall.png"
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print(f"Saved to: {output_path}")